In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BigDataAssignment") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.0


In [3]:
spark.range(1000000).count()

1000000

In [8]:
df = spark.read.parquet("/home/jovyan/data/yellow_tripdata_2025-01.parquet")

In [9]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
print(df.count())

3475226


In [11]:
print(len(df.columns))

20


In [12]:
df.show(20)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

In [ ]:
#Part 3 EDA

In [15]:
from pyspark.sql.functions import *

In [16]:
total_trips = df.count()
print("Total Trips:", total_trips)

Total Trips: 3475226


In [17]:
df.select(min("tpep_pickup_datetime").alias("Earliest Trip")).show()

+-------------------+
|      Earliest Trip|
+-------------------+
|2024-12-31 20:47:55|
+-------------------+



In [18]:
df.select(max("tpep_pickup_datetime").alias("Latest Trip")).show()

+-------------------+
|        Latest Trip|
+-------------------+
|2025-02-01 00:00:44|
+-------------------+



In [19]:
df.select("VendorID").distinct().show()

+--------+
|VendorID|
+--------+
|       1|
|       7|
|       2|
|       6|
+--------+



In [20]:
print("Unique Vendors:", df.select("VendorID").distinct().count())

Unique Vendors: 4


In [21]:
df.select(avg("trip_distance").alias("Average Distance")).show()

+-----------------+
| Average Distance|
+-----------------+
|5.855126178843539|
+-----------------+



In [22]:
df.select(avg("fare_amount").alias("Average Fare")).show()

+-----------------+
|     Average Fare|
+-----------------+
|17.08180276045484|
+-----------------+



In [23]:
df.select(max("fare_amount").alias("Maximum Fare")).show()

+------------+
|Maximum Fare|
+------------+
|   863372.12|
+------------+



In [24]:
df.select(min("fare_amount").alias("Minimum Fare")).show()

+------------+
|Minimum Fare|
+------------+
|      -900.0|
+------------+



In [25]:
df.select(avg("passenger_count").alias("Average Passengers")).show()

+------------------+
|Average Passengers|
+------------------+
|1.2978589658806226|
+------------------+



In [26]:
df.select("payment_type").distinct().show()

print("Payment Methods:",
      df.select("payment_type").distinct().count())

+------------+
|payment_type|
+------------+
|           5|
|           1|
|           3|
|           2|
|           4|
|           0|
+------------+

Payment Methods: 6


In [27]:
clean_df = df

In [ ]:
#part 4

In [28]:
print("Before:", clean_df.count())

Before: 3475226


In [29]:
clean_df = clean_df.dropDuplicates()

In [31]:
print("After:", clean_df.count())

After: 3475226


In [32]:
clean_df = clean_df.filter(col("trip_distance") > 0)

In [33]:
clean_df.select(min("trip_distance")).show()

+------------------+
|min(trip_distance)|
+------------------+
|              0.01|
+------------------+



In [34]:
clean_df = clean_df.filter(col("fare_amount") >= 0)

In [ ]:
clean_df.select(min("fare_amount")).show()

In [35]:
clean_df.select(
    [count(when(col(c).isNull(), c)).alias(c)
     for c in clean_df.columns]
).show(vertical=True)

-RECORD 0-----------------------
 VendorID              | 0      
 tpep_pickup_datetime  | 0      
 tpep_dropoff_datetime | 0      
 passenger_count       | 413422 
 trip_distance         | 0      
 RatecodeID            | 413422 
 store_and_fwd_flag    | 413422 
 PULocationID          | 0      
 DOLocationID          | 0      
 payment_type          | 0      
 fare_amount           | 0      
 extra                 | 0      
 mta_tax               | 0      
 tip_amount            | 0      
 tolls_amount          | 0      
 improvement_surcharge | 0      
 total_amount          | 0      
 congestion_surcharge  | 413422 
 Airport_fee           | 413422 
 cbd_congestion_fee    | 0      



In [ ]:
clean_df = clean_df.fillna(0)

In [36]:
clean_df.select(
    [count(when(col(c).isNull(), c)).alias(c)
     for c in clean_df.columns]
).show(vertical=True)

-RECORD 0-----------------------
 VendorID              | 0      
 tpep_pickup_datetime  | 0      
 tpep_dropoff_datetime | 0      
 passenger_count       | 413422 
 trip_distance         | 0      
 RatecodeID            | 413422 
 store_and_fwd_flag    | 413422 
 PULocationID          | 0      
 DOLocationID          | 0      
 payment_type          | 0      
 fare_amount           | 0      
 extra                 | 0      
 mta_tax               | 0      
 tip_amount            | 0      
 tolls_amount          | 0      
 improvement_surcharge | 0      
 total_amount          | 0      
 congestion_surcharge  | 413422 
 Airport_fee           | 413422 
 cbd_congestion_fee    | 0      



In [37]:
clean_df.cache()

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double]

In [ ]:
# part 5

In [38]:
high_fare = clean_df.filter(col("fare_amount") > 50)

high_fare.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-01-01 00:44:34|  2025-01-01 01:05:08|              1|        17.62|         1|                 N|         132|    

In [40]:
selected = clean_df.select(
    "VendorID",
    "passenger_count",
    "trip_distance",
    "fare_amount"
)

In [ ]:
selected.show(10)

In [41]:
tax_df = clean_df.withColumn(
    "fare_with_tax",
    round(col("fare_amount") * 1.10,2)
)

In [42]:
tax_df.select(
    "fare_amount",
    "fare_with_tax"
).show(10)

+-----------+-------------+
|fare_amount|fare_with_tax|
+-----------+-------------+
|       28.9|        31.79|
|        8.6|         9.46|
|       10.7|        11.77|
|        7.9|         8.69|
|       22.6|        24.86|
|       14.9|        16.39|
|       40.8|        44.88|
|       11.4|        12.54|
|       19.8|        21.78|
|       19.1|        21.01|
+-----------+-------------+
only showing top 10 rows



In [44]:
sorted_df = clean_df.orderBy(
    col("fare_amount").desc()
)

In [ ]:
sorted_df.show(10)

In [45]:
drop_df = clean_df.drop("congestion_surcharge")

drop_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [46]:
payment_df = clean_df.select(
    "payment_type"
).distinct()

In [ ]:
payment_df.show()

In [48]:
group_df = clean_df.groupBy(
    "payment_type"
).agg(
    round(avg("fare_amount"),2).alias("Average Fare")
)

In [ ]:
group_df.show()

In [50]:
alias_df = clean_df.select(
    col("fare_amount").alias("Fare"),
    col("trip_distance").alias("Distance")
)

In [ ]:
alias_df.show(10)

In [51]:
repart_df = clean_df.repartition(8)

print(
    repart_df.rdd.getNumPartitions()
)

8


In [52]:
payment_lookup = spark.createDataFrame(
[
    (0,"Unknown"),
    (1,"Credit Card"),
    (2,"Cash"),
    (3,"No Charge"),
    (4,"Dispute"),
    (5,"Unknown"),
    (6,"Voided Trip")
],
["payment_type","Payment_Name"])

In [53]:
join_df = clean_df.join(
    payment_lookup,
    on="payment_type",
    how="left"
)

In [ ]:
join_df.select(
    "payment_type",
    "Payment_Name",
    "fare_amount"
).show(10)

In [54]:
group_df.coalesce(1).write.mode("overwrite").option("header",True).csv("/home/jovyan/output/query1.csv")

In [ ]:
#part 6

In [55]:
clean_df.createOrReplaceTempView("taxi")

In [56]:
spark.sql("""
SELECT
    trip_distance,
    fare_amount,
    passenger_count
FROM taxi
ORDER BY trip_distance DESC
LIMIT 10
""").show()

+-------------+-----------+---------------+
|trip_distance|fare_amount|passenger_count|
+-------------+-----------+---------------+
|    276099.95|       9.13|           NULL|
|    222167.49|      31.19|           NULL|
|    206137.99|      24.89|           NULL|
|    202771.63|       10.1|           NULL|
|    189687.43|       12.7|           NULL|
|    181139.99|       6.33|           NULL|
|    164959.95|      14.05|           NULL|
|    158925.09|      10.82|           NULL|
|    156037.94|      28.74|           NULL|
|    143712.27|       8.51|           NULL|
+-------------+-----------+---------------+



In [57]:
spark.sql("""
SELECT
    PULocationID,
    COUNT(*) AS Total_Trips
FROM taxi
GROUP BY PULocationID
ORDER BY Total_Trips DESC
LIMIT 10
""").show()

+------------+-----------+
|PULocationID|Total_Trips|
+------------+-----------+
|         161|     161440|
|         237|     158123|
|         236|     149869|
|         132|     134879|
|         230|     118299|
|         186|     114159|
|         162|     112654|
|         142|     106071|
|         239|      91923|
|         163|      91585|
+------------+-----------+



In [58]:
spark.sql("""
SELECT
    payment_type,
    ROUND(AVG(fare_amount),2) AS Average_Fare
FROM taxi
GROUP BY payment_type
ORDER BY payment_type
""").show()

+------------+------------+
|payment_type|Average_Fare|
+------------+------------+
|           0|       18.14|
|           1|       17.85|
|           2|       18.02|
|           3|       18.31|
|           4|       45.64|
+------------+------------+



In [59]:
spark.sql("""
SELECT
    HOUR(tpep_pickup_datetime) AS Pickup_Hour,
    COUNT(*) AS Total_Trips
FROM taxi
GROUP BY Pickup_Hour
ORDER BY Total_Trips DESC
LIMIT 10
""").show()

+-----------+-----------+
|Pickup_Hour|Total_Trips|
+-----------+-----------+
|         18|     236644|
|         17|     229421|
|         15|     204571|
|         16|     203172|
|         19|     200689|
|         21|     194233|
|         14|     193993|
|         20|     184655|
|         13|     178588|
|         22|     170543|
+-----------+-----------+



In [60]:
spark.sql("""
SELECT
    VendorID,
    trip_distance,
    fare_amount
FROM taxi
WHERE trip_distance > 20
ORDER BY trip_distance DESC
""").show()

+--------+-------------+-----------+
|VendorID|trip_distance|fare_amount|
+--------+-------------+-----------+
|       2|    276099.95|       9.13|
|       2|    222167.49|      31.19|
|       2|    206137.99|      24.89|
|       2|    202771.63|       10.1|
|       2|    189687.43|       12.7|
|       2|    181139.99|       6.33|
|       2|    164959.95|      14.05|
|       2|    158925.09|      10.82|
|       2|    156037.94|      28.74|
|       2|    143712.27|       8.51|
|       2|    135116.83|      31.77|
|       2|    134033.15|      18.14|
|       2|    124083.23|      12.14|
|       2|    121799.97|      21.82|
|       2|    121555.16|       6.96|
|       2|    118435.89|      18.09|
|       2|    114364.71|       27.9|
|       2|    109183.02|      17.75|
|       2|    107806.31|      24.25|
|       2|    106629.95|      12.55|
+--------+-------------+-----------+
only showing top 20 rows



In [61]:
spark.sql("""
SELECT
    MONTH(tpep_pickup_datetime) AS Month,
    ROUND(SUM(total_amount),2) AS Revenue
FROM taxi
GROUP BY Month
ORDER BY Month
""").show()

+-----+-------------+
|Month|      Revenue|
+-----+-------------+
|    1|8.813264826E7|
|    2|        22.13|
|   12|       589.17|
+-----+-------------+



In [62]:
spark.sql("""
SELECT
    VendorID,
    ROUND(AVG(trip_distance),2) AS Avg_Distance
FROM taxi
GROUP BY VendorID
ORDER BY Avg_Distance DESC
""").show()

+--------+------------+
|VendorID|Avg_Distance|
+--------+------------+
|       6|         8.8|
|       2|        6.17|
|       1|        3.27|
|       7|        1.98|
+--------+------------+



In [63]:
spark.sql("""
SELECT
    passenger_count,
    ROUND(AVG(fare_amount),2) AS Avg_Fare
FROM taxi
GROUP BY passenger_count
ORDER BY passenger_count
""").show()

+---------------+--------+
|passenger_count|Avg_Fare|
+---------------+--------+
|           NULL|   18.14|
|              0|   15.49|
|              1|   17.77|
|              2|   20.16|
|              3|   19.85|
|              4|   22.18|
|              5|   16.85|
|              6|   17.34|
|              7|    79.0|
|              8|   69.73|
|              9|   91.33|
+---------------+--------+



In [64]:
spark.sql("""
SELECT
    fare_amount,
    trip_distance,
    payment_type
FROM taxi
ORDER BY fare_amount DESC
LIMIT 10
""").show()

+-----------+-------------+------------+
|fare_amount|trip_distance|payment_type|
+-----------+-------------+------------+
|  863372.12|          1.6|           4|
|     2450.9|       255.33|           2|
|     1309.2|       188.88|           2|
|      936.8|       143.54|           1|
|      900.0|          0.1|           3|
|     893.75|        133.3|           1|
|      826.2|       132.27|           4|
|      773.0|       119.66|           2|
|      700.0|         0.35|           4|
|      670.1|        111.2|           2|
+-----------+-------------+------------+



In [65]:
spark.sql("""
SELECT
ROUND(
AVG(
trip_distance /
(
(unix_timestamp(tpep_dropoff_datetime) -
unix_timestamp(tpep_pickup_datetime))/3600
)
),2
) AS Average_Speed_MPH
FROM taxi
WHERE unix_timestamp(tpep_dropoff_datetime) >
unix_timestamp(tpep_pickup_datetime)
""").show()

+-----------------+
|Average_Speed_MPH|
+-----------------+
|            24.12|
+-----------------+



In [66]:
sql_result = spark.sql("""
SELECT
payment_type,
ROUND(AVG(fare_amount),2) AS Average_Fare
FROM taxi
GROUP BY payment_type
""")

sql_result.coalesce(1)\
.write.mode("overwrite")\
.option("header",True)\
.csv("/home/jovyan/output/query2.csv")

In [ ]:
#part 7

In [67]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [68]:
windowSpec = Window.partitionBy("payment_type").orderBy(col("fare_amount").desc())

In [69]:
row_df = clean_df.withColumn(
    "row_number",
    row_number().over(windowSpec)
)

In [ ]:
row_df.select(
    "payment_type",
    "fare_amount",
    "row_number"
).show(20)

In [70]:
rank_df = clean_df.withColumn(
    "rank",
    rank().over(windowSpec)
)

In [ ]:
rank_df.select(
    "payment_type",
    "fare_amount",
    "rank"
).show(20)

In [71]:
dense_df = clean_df.withColumn(
    "dense_rank",
    dense_rank().over(windowSpec)
)

In [72]:
dense_df.select(
    "payment_type",
    "fare_amount",
    "dense_rank"
).show(20)

+------------+-----------+----------+
|payment_type|fare_amount|dense_rank|
+------------+-----------+----------+
|           0|     503.59|         1|
|           0|     300.63|         2|
|           0|     249.52|         3|
|           0|     235.37|         4|
|           0|      194.1|         5|
|           0|     191.39|         6|
|           0|     183.74|         7|
|           0|     172.35|         8|
|           0|     156.25|         9|
|           0|      154.2|        10|
|           0|      152.1|        11|
|           0|     150.29|        12|
|           0|     142.76|        13|
|           0|     137.57|        14|
|           0|      137.1|        15|
|           0|     129.81|        16|
|           0|     127.17|        17|
|           0|     125.03|        18|
|           0|      123.8|        19|
|           0|     121.56|        20|
+------------+-----------+----------+
only showing top 20 rows



In [73]:
dense_df.coalesce(1)\
.write.mode("overwrite")\
.option("header",True)\
.csv("/home/jovyan/output/query3.csv")

In [ ]:
# part 8

In [74]:
import time

start = time.time()

clean_df.groupBy("payment_type").count().show()

end = time.time()

print("Execution Time Before Cache:", end - start)

+------------+-------+
|payment_type|  count|
+------------+-------+
|           0| 413422|
|           1|2424734|
|           3|  12266|
|           2| 368511|
|           4|  35733|
+------------+-------+

Execution Time Before Cache: 1.1876485347747803


In [75]:
clean_df.cache()
clean_df.count()  # Materialize the cache

3254666

In [76]:
start = time.time()

clean_df.groupBy("payment_type").count().show()

end = time.time()

print("Execution Time After Cache:", end - start)

+------------+-------+
|payment_type|  count|
+------------+-------+
|           0| 413422|
|           1|2424734|
|           3|  12266|
|           2| 368511|
|           4|  35733|
+------------+-------+

Execution Time After Cache: 0.5685944557189941


In [77]:
repartition_df = clean_df.repartition(8)

print("Partitions:", repartition_df.rdd.getNumPartitions())

Partitions: 8


In [78]:
clean_df.groupBy("payment_type").count().explain(True)

== Parsed Logical Plan ==
'Aggregate ['payment_type], ['payment_type, count(1) AS count#11755L]
+- Filter (fare_amount#98 >= cast(0 as double))
   +- Filter (trip_distance#92 > cast(0 as double))
      +- Deduplicate [DOLocationID#96, improvement_surcharge#103, tpep_dropoff_datetime#90, PULocationID#95, trip_distance#92, Airport_fee#106, tolls_amount#102, RatecodeID#93L, VendorID#88, tip_amount#101, payment_type#97L, fare_amount#98, passenger_count#91L, store_and_fwd_flag#94, extra#99, cbd_congestion_fee#107, congestion_surcharge#105, total_amount#104, tpep_pickup_datetime#89, mta_tax#100]
         +- Relation [VendorID#88,tpep_pickup_datetime#89,tpep_dropoff_datetime#90,passenger_count#91L,trip_distance#92,RatecodeID#93L,store_and_fwd_flag#94,PULocationID#95,DOLocationID#96,payment_type#97L,fare_amount#98,extra#99,mta_tax#100,tip_amount#101,tolls_amount#102,improvement_surcharge#103,total_amount#104,congestion_surcharge#105,Airport_fee#106,cbd_congestion_fee#107] parquet

== Analyzed 

In [ ]:
#part 9